In [ ]:
"""
INTELLI-CREDIT | ML Layer — Notebook 4: Five Cs Score Aggregator
=================================================================
Run AFTER notebooks 1, 2, and 3.
CPU fine. Takes ~2 minutes.

WHAT THIS DOES:
  Reads all ML output CSVs and produces a single Five Cs scoring JSON
  per company — ready to be consumed by the FastAPI backend or displayed
  directly in the frontend scorecard (Screen 4).

INPUT FILES (from ml_outputs/):
  finbert_summary.csv
  news_risk_signals.csv
  isolation_forest_scores.csv
  peer_benchmark_narratives.csv
  graph_risk_scores.csv
  director_stress_scores.csv

ALSO READS (from data_u/):
  structured/companies_financial_scenarios.csv   (DSCR, sector etc.)
  unstructured/shareholding_pattern/promoter_pledge_analysis.csv
  external intelligence/legal_disputes/litigation_risk_summary_entity.csv
  primary insights/site_visit_cleaned.csv
  primary insights/management_interview_cleaned.csv

OUTPUT:
  five_cs_scores.csv           — flat per-company scores
  five_cs_scores_full.json     — full per-company JSON with all signals
  five_cs_ugro_demo.json       — Ugro Capital demo payload for frontend
"""

import json
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

BASE_PATH    = Path("/content/drive/Othercomputers/My PC/data_u")   # CHANGE THIS
ML_OUT       = Path("/content/drive/MyDrive/IntelliSense/ml_outputs")
OUTPUT_PATH  = ML_OUT   # save alongside other ML outputs

print("⏳ Loading all ML output files...")

# ── ML Outputs ────────────────────────────────────────────────────────────────
finbert_sum  = pd.read_csv(ML_OUT / "finbert_summary.csv") if (ML_OUT / "finbert_summary.csv").exists() else pd.DataFrame()
news_sigs    = pd.read_csv(ML_OUT / "news_risk_signals.csv") if (ML_OUT / "news_risk_signals.csv").exists() else pd.DataFrame()
iso_scores   = pd.read_csv(ML_OUT / "isolation_forest_scores.csv") if (ML_OUT / "isolation_forest_scores.csv").exists() else pd.DataFrame()
benchmarks   = pd.read_csv(ML_OUT / "peer_benchmark_narratives.csv") if (ML_OUT / "peer_benchmark_narratives.csv").exists() else pd.DataFrame()
graph_risk   = pd.read_csv(ML_OUT / "graph_risk_scores.csv") if (ML_OUT / "graph_risk_scores.csv").exists() else pd.DataFrame()
dir_stress   = pd.read_csv(ML_OUT / "director_stress_scores.csv") if (ML_OUT / "director_stress_scores.csv").exists() else pd.DataFrame()
fin_ratios   = pd.read_csv(ML_OUT / "computed_financial_ratios.csv") if (ML_OUT / "computed_financial_ratios.csv").exists() else pd.DataFrame()

print("⏳ Loading source data files...")

# ── Source Data ───────────────────────────────────────────────────────────────
fin_df       = pd.read_csv(BASE_PATH / "structured" / "companies_financial_scenarios.csv")
pledge_df    = pd.read_csv(BASE_PATH / "unstructured" / "shareholding_pattern" / "promoter_pledge_analysis.csv")
lit_df       = pd.read_csv(BASE_PATH / "external intelligence" / "legal_disputes" / "litigation_risk_summary_entity.csv") if (BASE_PATH / "external intelligence" / "legal_disputes" / "litigation_risk_summary_entity.csv").exists() else pd.DataFrame()
site_df      = pd.read_csv(BASE_PATH / "primary insights" / "site_visit_cleaned.csv") if (BASE_PATH / "primary insights" / "site_visit_cleaned.csv").exists() else pd.DataFrame()
mgmt_df      = pd.read_csv(BASE_PATH / "primary insights" / "management_interview_cleaned.csv") if (BASE_PATH / "primary insights" / "management_interview_cleaned.csv").exists() else pd.DataFrame()

print(f"✅ Loaded all inputs")

# ── Lookup helpers ─────────────────────────────────────────────────────────────
def get_row(df, company_id, id_col="company_id"):
    if df.empty or id_col not in df.columns:
        return None
    match = df[df[id_col] == company_id]
    return match.iloc[0] if not match.empty else None

def safe(val, default=0.0):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return default
    try:
        return float(val)
    except:
        return default

def clamp(val, lo=0.0, hi=10.0):
    return max(lo, min(hi, val))


# ============================================================
# SCORING FUNCTIONS — one per C
# ============================================================

def score_character_c(company_id: str, cin: str) -> dict:
    """
    Character C (0-10, higher = better character / lower risk)
    Weights:
      Litigation density (DRT/NCLT) : 20%
      FinBERT governance signals     : 20%
      Promoter pledge %              : 20%
      Graph risk (MCA + director)    : 15%
      Auditor / Going concern        : 15%  (not yet populated — use 0)
      GST / Circular trading flags   : 10%  (placeholder)
    """
    signals = []
    score   = 10.0   # start from perfect, apply penalties

    # 1. Litigation Density (weight 20%)
    lit_row = get_row(lit_df, company_id)
    if lit_row is not None:
        density_score = safe(lit_row.get("litigation_density_score"), 0)
        # density_score 0-10: 0=no litigation, 10=severe → invert for penalty
        penalty = (density_score / 10) * 2.0   # max penalty = 2.0 (20% of 10)
        score  -= penalty
        signals.append({
            "signal"    : "litigation_density",
            "value"     : round(density_score, 2),
            "penalty"   : round(penalty, 3),
            "feeds_into": "Character",
            "severity"  : "critical" if density_score > 7 else "high" if density_score > 4 else "medium",
        })
        # DRT = critical flag
        drt_count = safe(lit_row.get("drt_cases_active_count"), 0)
        if drt_count > 0:
            extra_penalty = min(2.0, drt_count * 1.0)
            score -= extra_penalty
            signals.append({"signal": "active_drt_cases", "value": drt_count,
                             "penalty": extra_penalty, "severity": "critical"})

    # 2. FinBERT governance signals (weight 20%)
    fb_row = get_row(finbert_sum, company_id)
    if fb_row is not None:
        finbert_risk = safe(fb_row.get("finbert_risk_score"), 0)
        penalty      = finbert_risk * 2.0   # max 2.0 if risk_score = 1.0
        score       -= penalty
        signals.append({
            "signal"    : "finbert_governance",
            "value"     : round(finbert_risk, 4),
            "penalty"   : round(penalty, 3),
            "severity"  : "high" if finbert_risk > 0.7 else "medium",
        })

    # 3. Promoter pledge % (weight 20%)
    # Get latest pledge from pledge_df
    if not pledge_df.empty:
        pledge_match = pledge_df[pledge_df["company_id"] == company_id].sort_values(
            "filing_date", ascending=False
        ) if "company_id" in pledge_df.columns else pd.DataFrame()
        if not pledge_match.empty:
            pledge_pct = safe(pledge_match.iloc[0].get("promoter_pledged_pct"), 0)
            risk_flag  = str(pledge_match.iloc[0].get("pledge_risk_flag", "normal"))
            pledge_penalty = {
                "normal": 0.0, "caution": 0.5, "high_risk": 1.2, "critical": 2.0
            }.get(risk_flag, 0.0)
            score -= pledge_penalty
            signals.append({
                "signal"      : "promoter_pledge_pct",
                "value"       : round(pledge_pct, 1),
                "risk_flag"   : risk_flag,
                "penalty"     : pledge_penalty,
                "severity"    : "critical" if risk_flag == "critical" else "high" if risk_flag == "high_risk" else "medium",
            })

    # 4. Graph risk (weight 15%)
    graph_row = get_row(graph_risk, company_id, "company_cin") if not graph_risk.empty else None
    if graph_row is None:
        graph_row = get_row(graph_risk, cin, "company_cin")
    if graph_row is not None:
        g_score   = safe(graph_row.get("graph_risk_score"), 0)
        penalty   = (g_score / 10) * 1.5   # max 1.5 = 15% weight
        score    -= penalty
        signals.append({
            "signal"    : "graph_risk",
            "value"     : round(g_score, 2),
            "penalty"   : round(penalty, 3),
            "severity"  : "critical" if g_score > 7 else "high" if g_score > 4 else "low",
        })

    # 5. Management credibility (from management interview)
    if not mgmt_df.empty and "case_id" in mgmt_df.columns:
        mgmt_match = mgmt_df[mgmt_df["company_id"] == company_id] if "company_id" in mgmt_df.columns else pd.DataFrame()
        if not mgmt_match.empty:
            credibility_map = {
                "confident_and_consistent": 0.0,
                "hesitant_but_reasonable" : 0.0,
                "evasive"                 : -0.8,
                "contradictory_to_documents": -1.5,
                "unable_to_explain"       : -1.0,
            }
            avg_adj = mgmt_match["management_credibility_assessment"].map(
                credibility_map
            ).mean() if "management_credibility_assessment" in mgmt_match.columns else 0.0
            score += safe(avg_adj, 0.0)   # negative values reduce score
            signals.append({"signal": "management_credibility", "adjustment": round(safe(avg_adj), 3)})

    char_score = clamp(score)
    return {
        "score"   : round(char_score, 2),
        "signals" : signals,
        "label"   : (
            "Critical Risk" if char_score < 4   else
            "High Risk"     if char_score < 6   else
            "Medium Risk"   if char_score < 7.5 else
            "Low Risk"
        ),
    }


def score_capacity_c(company_id: str) -> dict:
    """Capacity C — ability to repay from operations."""
    signals = []
    score   = 5.0   # start from mid-range, adjust from ratios

    # DSCR (most important)
    ratio_row = get_row(fin_ratios, company_id) if not fin_ratios.empty else None
    if ratio_row is not None:
        dscr = safe(ratio_row.get("dscr"), None)
        if dscr is not None:
            dscr_score = clamp(dscr * 3.5, 0, 10)   # DSCR 2.0+ → ~7, DSCR 1.0 → ~3.5
            score     += (dscr_score - 5.0) * 0.30   # 30% weight
            signals.append({"signal": "dscr", "value": round(dscr, 3), "score": round(dscr_score, 2)})

        ebitda_m = safe(ratio_row.get("ebitda_margin"), None)
        if ebitda_m is not None:
            m_score  = clamp(ebitda_m / 4.0, 0, 10)
            score   += (m_score - 5.0) * 0.20
            signals.append({"signal": "ebitda_margin", "value": round(ebitda_m, 2), "score": round(m_score, 2)})

        npa = safe(ratio_row.get("gross_npa_pct"), None)
        if npa is not None:
            npa_score = clamp(10 - npa, 0, 10)   # lower NPA = better
            score    += (npa_score - 5.0) * 0.20
            signals.append({"signal": "gross_npa_pct", "value": round(npa, 2), "score": round(npa_score, 2)})

    # Isolation Forest anomaly
    iso_row = get_row(iso_scores, company_id) if not iso_scores.empty else None
    if iso_row is not None:
        anom_score = safe(iso_row.get("anomaly_score_normalized"), 0.5)
        # anom_score 1=normal, 0=extreme_outlier → map to C adjustment
        adj = (anom_score - 0.5) * 2.0   # range: -1 to +1
        score += adj * 0.30
        signals.append({"signal": "isolation_forest", "value": round(anom_score, 4),
                         "label": str(iso_row.get("anomaly_classification", ""))})

    # Peer benchmarks (narrative signals)
    if not benchmarks.empty:
        cap_bench = benchmarks[
            (benchmarks["company_id"] == company_id) &
            (benchmarks["feeds_into_c"] == "Capacity") &
            (benchmarks["is_risk_signal"] == True)
        ]
        risk_signal_count = len(cap_bench)
        score -= risk_signal_count * 0.2
        if risk_signal_count > 0:
            signals.append({"signal": "peer_benchmark_risks", "count": risk_signal_count,
                             "narratives": cap_bench["narrative"].tolist()[:3]})

    return {
        "score"   : round(clamp(score), 2),
        "signals" : signals,
        "label"   : "Adequate" if clamp(score) >= 6 else "Stressed" if clamp(score) >= 4 else "Inadequate",
    }


def score_capital_c(company_id: str) -> dict:
    """Capital C — financial strength and leverage."""
    signals = []
    score   = 5.0

    ratio_row = get_row(fin_ratios, company_id) if not fin_ratios.empty else None
    if ratio_row is not None:
        de = safe(ratio_row.get("debt_to_equity"), None)
        if de is not None:
            # D/E: lower is better. 2x → score 7, 4x → 5, 6x → 3, >8 → 0
            de_score = clamp(10 - de * 1.0, 0, 10)
            score   += (de_score - 5.0) * 0.40
            signals.append({"signal": "debt_to_equity", "value": round(de, 3), "score": round(de_score, 2)})

        ic = safe(ratio_row.get("interest_coverage"), None)
        if ic is not None:
            ic_score = clamp(ic * 1.5, 0, 10)
            score   += (ic_score - 5.0) * 0.30
            signals.append({"signal": "interest_coverage", "value": round(ic, 3)})

    return {
        "score"   : round(clamp(score), 2),
        "signals" : signals,
        "label"   : "Strong" if clamp(score) >= 7 else "Moderate" if clamp(score) >= 5 else "Weak",
    }


def score_collateral_c(company_id: str, cin: str) -> dict:
    """Collateral C — quality and encumbrance of security."""
    signals = []
    score   = 6.0   # default moderate if no data

    # Live charge amount (encumbered collateral)
    graph_row = get_row(graph_risk, cin, "company_cin") if not graph_risk.empty else None
    if graph_row is not None:
        live_charges = safe(graph_row.get("live_charges_total_inr"), 0)
        if live_charges > 0:
            # More encumbrance = lower available collateral
            charge_cr = live_charges / 1e7   # convert to Cr
            penalty   = min(2.0, charge_cr / 500)   # >500Cr → max penalty
            score    -= penalty
            signals.append({"signal": "live_charges_total_cr", "value": round(charge_cr, 1),
                             "penalty": round(penalty, 2)})

    fin_row = get_row(fin_df, company_id) if "company_id" in fin_df.columns else None
    if fin_row is not None:
        # If collateral coverage ratio is in financial data
        ccr = safe(fin_row.get("collateral_coverage_ratio"), None)
        if ccr is not None:
            ccr_score = clamp(ccr * 3.0, 0, 10)
            score    += (ccr_score - 6.0) * 0.50
            signals.append({"signal": "collateral_coverage_ratio", "value": round(ccr, 2)})

    return {
        "score"   : round(clamp(score), 2),
        "signals" : signals,
        "label"   : "Adequate" if clamp(score) >= 6 else "Insufficient",
    }


def score_conditions_c(company_id: str, sector: str) -> dict:
    """Conditions C — external environment, macro, sector."""
    signals = []
    score   = 6.0   # default neutral macro

    # News sector signals
    if not news_sigs.empty and "company_id" in news_sigs.columns:
        sector_news = news_sigs[
            (news_sigs["company_id"] == company_id) &
            (news_sigs["signal_category"].str.contains("sector_", na=False))
        ]
        sector_risk_count  = len(sector_news[sector_news.get("is_high_severity", False)])
        score -= sector_risk_count * 0.3
        if sector_risk_count > 0:
            signals.append({"signal": "sector_news_risks", "count": sector_risk_count})

    # Isolation Forest peer position (sector context)
    iso_row = get_row(iso_scores, company_id) if not iso_scores.empty else None
    if iso_row is not None:
        anom_class = str(iso_row.get("anomaly_classification", "normal"))
        adj = {"normal": 0.5, "anomalous": -0.5, "extreme_outlier": -1.5}.get(anom_class, 0)
        score += adj
        signals.append({"signal": "peer_position", "label": anom_class, "adjustment": adj})

    # Condition benchmarks
    if not benchmarks.empty:
        cond_bench = benchmarks[
            (benchmarks["company_id"] == company_id) &
            (benchmarks["feeds_into_c"] == "Conditions") &
            (benchmarks["is_risk_signal"] == True)
        ]
        score -= len(cond_bench) * 0.2

    return {
        "score"   : round(clamp(score), 2),
        "signals" : signals,
        "sector"  : sector,
        "label"   : "Favourable" if clamp(score) >= 7 else "Neutral" if clamp(score) >= 5 else "Adverse",
    }


def compute_composite(char: float, cap: float, capit: float, coll: float, cond: float,
                       weights: dict = None) -> float:
    """Weighted composite score. Default weights from methodology."""
    w = weights or {
        "Character" : 0.30,
        "Capacity"  : 0.30,
        "Capital"   : 0.20,
        "Collateral": 0.10,
        "Conditions": 0.10,
    }
    return round(
        char * w["Character"] + cap * w["Capacity"] + capit * w["Capital"] +
        coll * w["Collateral"] + cond * w["Conditions"], 2
    )

def get_decision(score: float) -> str:
    if score >= 7.5:  return "APPROVE"
    if score >= 6.0:  return "APPROVE WITH CONDITIONS"
    if score >= 5.0:  return "REFER TO SENIOR CREDIT COMMITTEE"
    return "REJECT"


# ============================================================
# RUN SCORING FOR ALL COMPANIES
# ============================================================
print("\n⏳ Computing Five Cs scores for all companies...")

all_companies = fin_df[["case_id", "company_id", "company_cin", "sector"]].drop_duplicates()
results_flat  = []
results_full  = {}

for _, row in all_companies.iterrows():
    cid    = str(row.get("company_id", ""))
    cin    = str(row.get("company_cin", ""))
    sector = str(row.get("sector", "Other"))
    case   = str(row.get("case_id", ""))

    char  = score_character_c(cid, cin)
    cap   = score_capacity_c(cid)
    capit = score_capital_c(cid)
    coll  = score_collateral_c(cid, cin)
    cond  = score_conditions_c(cid, sector)

    composite = compute_composite(
        char["score"], cap["score"], capit["score"],
        coll["score"], cond["score"]
    )
    decision = get_decision(composite)

    results_flat.append({
        "case_id"         : case,
        "company_id"      : cid,
        "company_cin"     : cin,
        "sector"          : sector,
        "character_score" : char["score"],
        "capacity_score"  : cap["score"],
        "capital_score"   : capit["score"],
        "collateral_score": coll["score"],
        "conditions_score": cond["score"],
        "composite_score" : composite,
        "decision"        : decision,
        "character_label" : char["label"],
        "capacity_label"  : cap["label"],
        "capital_label"   : capit["label"],
        "collateral_label": coll["label"],
        "conditions_label": cond["label"],
        "computed_at"     : datetime.utcnow().isoformat(),
    })

    results_full[cid] = {
        "case_id"  : case,
        "company_id": cid,
        "company_cin": cin,
        "sector"   : sector,
        "composite_score": composite,
        "decision" : decision,
        "weights"  : {"Character": 0.30, "Capacity": 0.30, "Capital": 0.20, "Collateral": 0.10, "Conditions": 0.10},
        "five_cs"  : {
            "Character" : char,
            "Capacity"  : cap,
            "Capital"   : capit,
            "Collateral": coll,
            "Conditions": cond,
        },
        "peer_benchmarks": benchmarks[benchmarks["company_id"] == cid].to_dict("records") if not benchmarks.empty else [],
        "computed_at": datetime.utcnow().isoformat(),
    }

# Save flat CSV
flat_df = pd.DataFrame(results_flat)
flat_out = OUTPUT_PATH / "five_cs_scores.csv"
flat_df.to_csv(flat_out, index=False)
print(f"💾 Saved: {flat_out}  ({len(flat_df)} companies scored)")

# Save full JSON
full_out = OUTPUT_PATH / "five_cs_scores_full.json"
with open(full_out, "w") as f:
    json.dump(results_full, f, indent=2, default=str)
print(f"💾 Saved: {full_out}")

# ── Ugro Capital demo payload ──────────────────────────────────────────────────
ugro_key = next(
    (k for k in results_full if "ugro" in k.lower()),
    list(results_full.keys())[0] if results_full else None
)

if ugro_key:
    ugro_payload = results_full[ugro_key]
    ugro_out = OUTPUT_PATH / "five_cs_ugro_demo.json"
    with open(ugro_out, "w") as f:
        json.dump(ugro_payload, f, indent=2, default=str)
    print(f"💾 Saved: {ugro_out}  ← demo payload for Ugro Capital")

    print("\n" + "="*60)
    print("UGRO CAPITAL — FINAL FIVE Cs SCORECARD")
    print("="*60)
    p = ugro_payload
    print(f"  {'Character C':20s}: {p['five_cs']['Character']['score']:4.1f} / 10  [{p['five_cs']['Character']['label']}]")
    print(f"  {'Capacity C':20s}: {p['five_cs']['Capacity']['score']:4.1f} / 10  [{p['five_cs']['Capacity']['label']}]")
    print(f"  {'Capital C':20s}: {p['five_cs']['Capital']['score']:4.1f} / 10  [{p['five_cs']['Capital']['label']}]")
    print(f"  {'Collateral C':20s}: {p['five_cs']['Collateral']['score']:4.1f} / 10  [{p['five_cs']['Collateral']['label']}]")
    print(f"  {'Conditions C':20s}: {p['five_cs']['Conditions']['score']:4.1f} / 10  [{p['five_cs']['Conditions']['label']}]")
    print(f"  {'─'*40}")
    print(f"  {'COMPOSITE SCORE':20s}: {p['composite_score']:4.1f} / 10")
    print(f"  {'DECISION':20s}: {p['decision']}")

print(f"\n✅ Five Cs Aggregator complete.")
print(f"""
BACKEND INTEGRATION:
  Import five_cs_scores.csv into your PostgreSQL (or serve via FastAPI)
  five_cs_scores_full.json → GET /api/case/{{case_id}}/scores
  five_cs_ugro_demo.json   → preloaded demo payload for judges

DECISION THRESHOLDS:
  ≥ 7.5  → APPROVE
  6.0–7.5 → APPROVE WITH CONDITIONS
  5.0–6.0 → REFER TO SENIOR CREDIT COMMITTEE
  < 5.0   → REJECT
""")

⏳ Loading all ML output files...
⏳ Loading source data files...
✅ Loaded all inputs

⏳ Computing Five Cs scores for all companies...
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/five_cs_scores.csv  (2238 companies scored)
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/five_cs_scores_full.json
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/five_cs_ugro_demo.json  ← demo payload for Ugro Capital

UGRO CAPITAL — FINAL FIVE Cs SCORECARD
  Character C         :  8.1 / 10  [Low Risk]
  Capacity C          :  5.0 / 10  [Stressed]
  Capital C           :  5.0 / 10  [Moderate]
  Collateral C        :  6.0 / 10  [Insufficient]
  Conditions C        :  5.2 / 10  [Neutral]
  ────────────────────────────────────────
  COMPOSITE SCORE     :  6.0 / 10
  DECISION            : APPROVE WITH CONDITIONS

✅ Five Cs Aggregator complete.

BACKEND INTEGRATION:
  Import five_cs_scores.csv into your PostgreSQL (or serve via FastAPI)
  five_cs_scores_full.json → GET /api/case/{case_i